In [1]:
# ==========================================
# 1. Carga segura de credenciales e imports
# ==========================================
import os
from dotenv import load_dotenv

load_dotenv()

import json
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QAOAAnsatz
from qiskit.primitives import StatevectorEstimator, StatevectorSampler
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.converters import LinearEqualityToPenalty
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_algorithms import NumPyMinimumEigensolver
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    SamplerV2 as Sampler,
)

IBM_TOKEN = os.getenv("QISKIT_IBM_TOKEN")
IBM_CRN   = os.getenv("QISKIT_IBM_CRN")

if IBM_TOKEN:
    print(f"✓ Token cargado ({len(IBM_TOKEN)} caracteres)")
else:
    print("⚠ No hay QISKIT_IBM_TOKEN en .env — se usarán simuladores locales")

✓ Token cargado (44 caracteres)


In [2]:
# ==========================================
# 2. Conexión opcional a IBM Quantum
# ==========================================
USE_REAL_HARDWARE = bool(IBM_TOKEN)

if USE_REAL_HARDWARE:
    try:
        service = QiskitRuntimeService(
            channel="ibm_quantum_platform",
            token=IBM_TOKEN,
            instance=IBM_CRN,
        )
        backend = service.least_busy(operational=True, simulator=False, min_num_qubits=156)
        print("✓ Conectado a IBM Quantum")
        print(f"  Backend: {backend.name} ({backend.num_qubits} qubits)")
        print(f"  Cola estimada: {backend.status().pending_jobs} jobs pendientes")
    except Exception as e:
        print(f"✗ Error de conexión: {e}")
        print("  → Usando simulador local")
        USE_REAL_HARDWARE = False
        backend = None
else:
    backend = None
    print("→ Modo simulador local")

qiskit_runtime_service._discover_account:WARNING:2026-11-05 17:37:11,334: Loading account with the given token. A saved account will not be used.


✓ Conectado a IBM Quantum
  Backend: ibm_basquecountry (156 qubits)
  Cola estimada: 8 jobs pendientes


In [3]:
# ==========================================
# 3. Carga de datos del portafolio
# ==========================================
with open("../data/intermediate/parametros_16_activos.json", "r") as f:
    datos = json.load(f)

n = datos["n_activos"]
tickers = datos["tickers"]
mu = np.array(datos["mu"])
sigma = np.array(datos["sigma"])

K = 4
q = 0.5
lam = 10.0

print(f"Optimizando portafolio para {n} activos: {tickers}")

Optimizando portafolio para 16 activos: ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'META', 'TSLA', 'IBE.MC', 'BBVA.MC', 'CAF.MC', 'CIE.MC', 'SAN.MC', 'TEF.MC', 'REP.MC', 'ITX.MC', 'AENA.MC']


In [4]:
# ==========================================
# 4. Construcción del QP/QUBO
# ==========================================
def build_portfolio_qp(penalty_lam):
    qp = QuadraticProgram()
    for ticker in tickers:
        qp.binary_var(name=ticker)
    qp.maximize(linear=mu, quadratic=-q * sigma)
    restriccion = {ticker: 1 for ticker in tickers}
    qp.linear_constraint(linear=restriccion, sense="==", rhs=K, name="budget")
    converter = LinearEqualityToPenalty(penalty=penalty_lam)
    qubo = converter.convert(qp)
    return qp, qubo

In [5]:
# ==========================================
# 5. Solución exacta (NumPyMinimumEigensolver)
# ==========================================
qp, qubo = build_portfolio_qp(penalty_lam=lam)
cost_hamiltonian, offset = qubo.to_ising()

exact_optimizer = MinimumEigenOptimizer(NumPyMinimumEigensolver())
exact_result = exact_optimizer.solve(qubo)
optimal_value_exact = exact_result.fval
exact_x = [int(val) for val in exact_result.x]
exact_activos = [tickers[i] for i in range(n) if exact_x[i] == 1]

print("--- SOLUCIÓN EXACTA ---")
print(f"Valor óptimo teórico: {optimal_value_exact:.4f}")
print(f"Vector binario: {exact_x}")
print(f"Activos: {exact_activos}")
print("-----------------------")

--- SOLUCIÓN EXACTA ---
Valor óptimo teórico: 2.5290
Vector binario: [0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0]
Activos: ['GOOGL', 'BBVA.MC', 'CAF.MC', 'SAN.MC']
-----------------------


In [6]:
# ==========================================
# 6. Baseline Greedy
# ==========================================
def evaluate_portfolio(x_vec):
    x_arr = np.array(x_vec)
    return float(np.dot(mu, x_arr) - q * np.dot(x_arr, np.dot(sigma, x_arr)))

selected_greedy = []
greedy_x = np.zeros(n, dtype=int)

for _ in range(K):
    best_v = None
    best_win = -np.inf
    for i in range(n):
        if i not in selected_greedy:
            candidate = greedy_x.copy()
            candidate[i] = 1
            gain = evaluate_portfolio(candidate)
            if gain > best_win:
                best_win = gain
                best_v = i
    selected_greedy.append(best_v)
    greedy_x[best_v] = 1

greedy_value = evaluate_portfolio(greedy_x)
greedy_activos = [tickers[i] for i in range(n) if greedy_x[i] == 1]

print("--- SOLUCIÓN GREEDY ---")
print(f"Valor óptimo greedy: {greedy_value:.4f}")
print(f"Vector binario: {greedy_x.tolist()}")
print(f"Activos: {greedy_activos}")
print("-----------------------")

--- SOLUCIÓN GREEDY ---
Valor óptimo greedy: 2.5290
Vector binario: [0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0]
Activos: ['GOOGL', 'BBVA.MC', 'CAF.MC', 'SAN.MC']
-----------------------


In [7]:
# ==========================================
# 7. Fuerza bruta exacta + Greedy con swaps
# ==========================================
from itertools import combinations

best_val, best_x = -np.inf, None
for comb in combinations(range(n), K):
    x = np.zeros(n, dtype=int)
    x[list(comb)] = 1
    v = evaluate_portfolio(x)
    if v > best_val:
        best_val, best_x = v, x

print(f"Exacto: {best_val:.4f}", best_x.tolist(), "factible:", best_x.sum() == K)

def local_search(x0):
    x = x0.copy()
    cur = evaluate_portfolio(x)
    improved = True
    while improved:
        improved = False
        for i in np.where(x == 1)[0]:
            for j in np.where(x == 0)[0]:
                y = x.copy()
                y[i], y[j] = 0, 1
                v = evaluate_portfolio(y)
                if v > cur + 1e-12:
                    x, cur, improved = y, v, True
                    break
            if improved:
                break
    return x, cur

ls_x, ls_value = local_search(greedy_x)
print(f"Greedy+swap: {ls_value:.4f}", ls_x.tolist())
print(f"Ratio greedy: {greedy_value/best_val:.4f} | greedy+swap: {ls_value/best_val:.4f}")

Exacto: 2.5290 [0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0] factible: True
Greedy+swap: 2.5290 [0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0]
Ratio greedy: 1.0000 | greedy+swap: 1.0000


In [8]:
# ==========================================
# 8. Relajación continua + Warm-Start (theta, mixer)
# ==========================================
def solve_continuous_relaxation(mu, sigma, q, K):
    n_assets = len(mu)
    def obj(x):
        return -(np.dot(mu, x) - q * np.dot(x, np.dot(sigma, x)))
    def grad(x):
        return -(mu - 2 * q * np.dot(sigma, x))
    cons = {'type': 'eq', 'fun': lambda x: np.sum(x) - K}
    bounds = [(0.0, 1.0) for _ in range(n_assets)]
    x0 = np.full(n_assets, K / n_assets)
    res = minimize(obj, x0, method='SLSQP', jac=grad, bounds=bounds, constraints=cons)
    return res.x

c_stars = solve_continuous_relaxation(mu, sigma, q, K)
print("Solución relajada continua c*:", np.round(c_stars, 3))
print("Top K índices continuos:", np.argsort(c_stars)[::-1][:K].tolist())

eps = 0.25
c_reg = np.clip(c_stars, eps, 1.0 - eps)
thetas = 2.0 * np.arcsin(np.sqrt(c_reg))

ws_init_qc = QuantumCircuit(n)
for i, theta in enumerate(thetas):
    ws_init_qc.ry(theta, i)

pauli_list = []
for i, theta in enumerate(thetas):
    x_str = ['I'] * n
    z_str = ['I'] * n
    x_str[n - 1 - i] = 'X'
    z_str[n - 1 - i] = 'Z'
    pauli_list.append(("".join(x_str), -np.sin(theta)))
    pauli_list.append(("".join(z_str), -np.cos(theta)))

ws_mixer = SparsePauliOp.from_list(pauli_list)

Solución relajada continua c*: [0. 0. 0. 1. 0. 0. 0. 0. 1. 1. 0. 1. 0. 0. 0. 0.]
Top K índices continuos: [9, 3, 8, 11]


In [9]:
# ==========================================
# 9. Motor QAOA híbrido
#    - Optimización de ángulos: SIEMPRE local (StatevectorEstimator)
#    - Muestreo final: en QPU real solo si use_hw=True
#    - Ansatz pre-descompuesto UNA VEZ por run (clave para velocidad)
# ==========================================
def run_qaoa_experiment(reps=1, warm_start=False, maxiter=100, seed=42,
                        use_hw=None, shots=4096, verbose=True):
    if use_hw is None:
        use_hw = USE_REAL_HARDWARE

    np.random.seed(seed)

    # 1. Construir ansatz abstracto
    if warm_start:
        ansatz = QAOAAnsatz(
            cost_operator=cost_hamiltonian, reps=reps,
            initial_state=ws_init_qc, mixer_operator=ws_mixer,
        )
        init_params = np.random.uniform(-0.05, 0.05, ansatz.num_parameters)
    else:
        ansatz = QAOAAnsatz(cost_operator=cost_hamiltonian, reps=reps)
        init_params = np.random.uniform(0, np.pi, ansatz.num_parameters)

    # 2. Pre-descomponer UNA SOLA VEZ (acelera ~50x por iteración)
    ansatz_local = transpile(ansatz, basis_gates=["u", "cx"], optimization_level=0)

    local_estimator = StatevectorEstimator()
    history = []

    def cost_func(params):
        pub = (ansatz_local, cost_hamiltonian, params)
        job = local_estimator.run([pub])
        energy = float(job.result()[0].data.evs)
        history.append(energy + offset)
        if verbose and len(history) % 25 == 0:
            print(f"      eval {len(history):3d} | E+offset = {energy + offset:.4f}")
        return energy

    res = minimize(
        cost_func, init_params, method="COBYLA",
        options={"maxiter": maxiter, "rhobeg": 0.15, "tol": 1e-3},
    )

    # 3. Circuito final con ángulos óptimos
    qc_measured = ansatz.assign_parameters(res.x)
    qc_measured.measure_all()

    # 4. Muestreo (QPU real o local)
    if use_hw and backend is not None:
        pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
        isa_circuit = pm.run(qc_measured)
        sampler = Sampler(mode=backend)
        samp_job = sampler.run([isa_circuit], shots=shots)
        counts = samp_job.result()[0].data.meas.get_counts()
        print(f"      [QPU] job_id = {samp_job.job_id()}")
    else:
        local_sampler = StatevectorSampler()
        samp_job = local_sampler.run([(qc_measured, None)], shots=shots)
        counts = samp_job.result()[0].data.meas.get_counts()

    total_shots = sum(counts.values())

    # 5. Métricas del portafolio
    feasible_prob = 0.0
    expected_feasible_val = 0.0
    best_sampled_val = -np.inf
    best_sampled_x = None
    most_frequent_feasible_x = None
    max_feasible_counts = -1

    for bitstring, count in counts.items():
        x_vec = np.array([int(b) for b in bitstring[::-1]], dtype=int)
        prob = count / total_shots
        if np.sum(x_vec) == K:
            feasible_prob += prob
            val = evaluate_portfolio(x_vec)
            expected_feasible_val += prob * val
            if val > best_sampled_val:
                best_sampled_val = val
                best_sampled_x = x_vec
            if count > max_feasible_counts:
                max_feasible_counts = count
                most_frequent_feasible_x = x_vec

    mean_val_given_feasible = (
        expected_feasible_val / feasible_prob if feasible_prob > 0 else np.nan
    )
    mode_val = (
        evaluate_portfolio(most_frequent_feasible_x)
        if most_frequent_feasible_x is not None else np.nan
    )

    return {
        "res": res,
        "history": history,
        "n_evals": len(history),
        "feasible_prob": feasible_prob,
        "mean_val_feasible": mean_val_given_feasible,
        "mode_val": mode_val,
        "mode_x": most_frequent_feasible_x,
        "best_sampled_val": best_sampled_val,
        "best_sampled_x": best_sampled_x,
        "used_hardware": use_hw,
    }

In [ ]:
# ==========================================
# 10. Barrido de p=1,2,3 — SOLO SIMULADOR
#     (con timing por run y maxiter reducido)
# ==========================================
depths = [1, 2, 3]
results_std = {}
results_ws = {}
timings = {}

for p in depths:
    print(f"\n{'='*70}")
    print(f"  p = {p}")
    print(f"{'='*70}")

    print("  [Standard QAOA]")
    t0 = time.time()
    results_std[p] = run_qaoa_experiment(reps=p, warm_start=False,
                                          maxiter=100, use_hw=True, verbose=True)
    t_std = time.time() - t0
    print(f"  → Completado en {t_std:.1f}s ({results_std[p]['n_evals']} evals)")

    print("  [Warm-Start QAOA]")
    t0 = time.time()
    results_ws[p] = run_qaoa_experiment(reps=p, warm_start=True,
                                         maxiter=100, use_hw=True, verbose=True)
    t_ws = time.time() - t0
    print(f"  → Completado en {t_ws:.1f}s ({results_ws[p]['n_evals']} evals)")

    timings[p] = (t_std, t_ws)

# --- Resumen ---
print("\n" + "="*80)
print(f"Óptimo Exacto (Brute Force) : {best_val:.4f} (Ratio = 1.0000)")
print(f"Baseline Greedy Clásico     : {greedy_value:.4f} (Ratio = {greedy_value/best_val:.4f})")
print("="*80)
print(f"{'Método':<18} | {'p':<2} | {'P(Fact.)':<10} | {'Ratio Moda':<11} | {'Ratio Media':<11} | {'t (s)':<6}")
print("-"*80)

for p in depths:
    r_s = results_std[p]
    r_w = results_ws[p]
    print(f"{'Standard QAOA':<18} | {p:<2} | {r_s['feasible_prob']*100:>8.2f}% | {r_s['mode_val']/best_val:>11.4f} | {r_s['mean_val_feasible']/best_val:>11.4f} | {timings[p][0]:>6.1f}")
    print(f"{'Warm-Start QAOA':<18} | {p:<2} | {r_w['feasible_prob']*100:>8.2f}% | {r_w['mode_val']/best_val:>11.4f} | {r_w['mean_val_feasible']/best_val:>11.4f} | {timings[p][1]:>6.1f}")
    print("-"*80)

best_p = max(depths, key=lambda p: results_ws[p]['mode_val'])
ws_winner_x = results_ws[best_p]['mode_x']
ws_winner_tickers = [tickers[i] for i in range(n) if ws_winner_x[i] == 1]
print(f"\nMejor portafolio (Moda Warm-Start p={best_p}): {ws_winner_tickers}")
print(f"Valor: {results_ws[best_p]['mode_val']:.4f} | "
      f"Supera al Greedy ({greedy_value:.4f}): {results_ws[best_p]['mode_val'] >= greedy_value}")


  p = 1
  [Standard QAOA]


/tmp/ipykernel_107671/1702893443.py:53: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  sampler = Sampler(mode=backend)


      [QPU] job_id = dbmbeto21fus73bmk01g
  → Completado en 146.4s (16 evals)
  [Warm-Start QAOA]
      [QPU] job_id = dbmbg2g21fus73bmk12g
  → Completado en 18.9s (18 evals)

  p = 2
  [Standard QAOA]
      eval  25 | E+offset = 122.6146
      [QPU] job_id = dbmbg88suuec73fjlvpg
  → Completado en 78.0s (34 evals)
  [Warm-Start QAOA]
